# S&P 500 — Pipeline de Construção da Base de Dados
## TCC: Pair Trading

Este notebook organiza o pipeline completo de construção da base de preços do S&P 500,
livre de viés de sobrevivência, para uso no TCC de pair trading.

---

### Fluxo geral

```
ETAPA 1 │ Importações e configuração de caminhos
ETAPA 2 │ Carregar a base original do professor (1990/S2 – 2015/S1)
ETAPA 3 │ Carregar o Periods.csv e construir o mapeamento semestral
ETAPA 4 │ Unir base de preços + mapeamento de períodos → sp500_with_periods
ETAPA 5 │ Carregar e validar os constituintes históricos do S&P 500
ETAPA 6 │ Gerar snapshots semestrais (2015/S1 – 2025/S2) e lista de tickers
ETAPA 7 │ Coletar preços via Yahoo Finance (com checkpoint de retomada)
```

> **Sobre viés de sobrevivência:** a base inclui todas as empresas que já fizeram
> parte do índice, mesmo as que faliram, foram adquiridas ou foram removidas.
> Isso é essencial para evitar que os resultados do backtest sejam inflados por
> considerar apenas as empresas que "sobreviveram".


---
## Etapa 1 — Importações e configuração de caminhos

Centralizar todos os caminhos de arquivo aqui facilita mover o projeto entre
máquinas: basta ajustar `BASE_DIR`.


In [80]:
import pandas as pd
import numpy as np
import yfinance as yf
import time
from pathlib import Path

print(f"pandas   {pd.__version__}")
print(f"yfinance {yf.__version__}")

# Nota sobre versões do yfinance:
# Em versões recentes (≥ 0.2.x), ao usar auto_adjust=False, o retorno inclui
# as colunas: Open, High, Low, Close, Adj Close, Volume
# A coluna 'Close' é o preço sem ajuste de dividendos (split-adjusted apenas)
# A coluna 'Adj Close' inclui ajuste de dividendos — NÃO usar para esta base


pandas   2.2.3
yfinance 1.2.0


In [81]:
# ── Caminho raiz do projeto ───────────────────────────────────────────────────
# Ajuste aqui se mover os arquivos de lugar.
BASE_DIR        = Path(r"C:\Users\jvlei\Desktop\TCC-pair-trading\data_bases")
BASE_DIR_OUTPUT = BASE_DIR / "output"

# ── Arquivos de entrada ───────────────────────────────────────────────────────
PATH_DB       = BASE_DIR / "Pt_formatted.csv"                                   # Base do professor (fatia de 4k linhas para exploração)
PATH_DB_FULL  = BASE_DIR / "Pt_formatted.csv"                             # Base completa (6426 linhas) — usar para validação de datas
PATH_PERIODS  = BASE_DIR / "Periods.csv"                                        # Dias de pregão por semestre
PATH_SP500HIS = BASE_DIR / "S&P 500 Historical Components & Changes(01-17-2026).csv"  # Constituintes históricos

# ── Arquivos de saída (gerados por este notebook) ─────────────────────────────
PATH_DB_PERIODS   = BASE_DIR_OUTPUT / "sp500_with_periods.csv"                  # Base + metadados de período
PATH_SNAPSHOTS    = BASE_DIR_OUTPUT / "snapshots_semestrais.csv"                # Constituintes por semestre
PATH_TICKERS      = BASE_DIR_OUTPUT / "tickers_a_coletar.csv"                   # Tickers únicos + janela de coleta
PATH_REPORT       = BASE_DIR_OUTPUT / "coleta_report.csv"                       # Status da coleta (ok/delisted/erro)
DIR_PRICES        = BASE_DIR_OUTPUT / "prices"                                   # Um CSV por ticker

DIR_PRICES.mkdir(parents=True, exist_ok=True)

print("Caminhos configurados:")
for name, p in [
    ("Base do professor", PATH_DB),
    ("Periods.csv",       PATH_PERIODS),
    ("Constituintes S&P", PATH_SP500HIS),
]:
    status = "✅ existe" if p.exists() else "❌ NÃO encontrado"
    print(f"  {status}  {p.name}")


Caminhos configurados:
  ✅ existe  Pt_formatted.csv
  ✅ existe  Periods.csv
  ✅ existe  S&P 500 Historical Components & Changes(01-17-2026).csv


---
## Etapa 2 — Base original do professor (1990/S2 – 2015/S1)

Formato wide: cada linha é um dia de pregão, cada coluna é um ticker.
Os valores são preços de fechamento ajustados por splits.
Empresas que faliram ou foram removidas do índice mantêm seus dados
históricos — identificáveis pelo sufixo `Q` no ticker (ex: `ENRNQ` = Enron).


In [82]:
db = pd.read_csv(PATH_DB)

print(f"Shape: {db.shape[0]:,} linhas × {db.shape[1]:,} colunas")
print(f"Primeiros tickers: {db.columns[:8].tolist()}")
print(f"Últimos  tickers: {db.columns[-5:].tolist()}")


Shape: 6,426 linhas × 1,100 colunas
Primeiros tickers: ['UN', '1005945D', '162007Q', 'RDPL', 'BGG', 'SKY', 'GOSHA', 'HSY']
Últimos  tickers: ['EQIX', '1436513D', '1431816D', 'SPGI', '1448062D']


In [83]:
# Inspeção rápida: empresas famosas para verificar que os valores fazem sentido.
# Todos os valores são split-adjusted, então AAPL em 1990 aparece como ~$1.
spot_check = {
    "AAPL": "Apple — ~$1 em 1990 (split-adjusted)",
    "GE":   "GE — presente desde o início",
    "XOM":  "ExxonMobil",
}
for ticker, nota in spot_check.items():
    if ticker in db.columns:
        primeiros = db[ticker].dropna().head(3).round(4).tolist()
        print(f"{ticker:6s} ({nota}): {primeiros}")


AAPL   (Apple — ~$1 em 1990 (split-adjusted)): [1.3753, 1.3753, 1.3596]
GE     (GE — presente desde o início): [2.8747, 2.9053, 2.8645]
XOM    (ExxonMobil): [5.6624, 5.6328, 5.5438]


---
## Etapa 3 — Mapeamento de períodos (Periods.csv)

`Periods.csv` contém a contagem oficial de dias de pregão da NYSE por semestre,
fornecida pelo professor. Essa contagem é usada como **fonte verdadeira** para
mapear cada linha da base a um período — não usamos aproximações de calendário.

**Estrutura dos 51 semestres:**
- `idx 0`  → `1990/S2` (jul–dez 1990) ← a base começa no 2º semestre de 1990
- `idx 1`  → `1991/S1` (jan–jun 1991)
- `idx 2`  → `1991/S2` ...
- `idx 50` → `2015/S1` (jan–jun 2015) ← último semestre da base do professor

**Âncoras de validação:**
- `2001/S2` tem **123 dias** (único semestre <124) → reflexo do fechamento da NYSE após o 11/set
- `GOOGL` aparece pela primeira vez na linha **3.563** → cai em `2004/S2` (IPO ago/2004)


In [84]:
# Carregar sem cabeçalho (a primeira linha já é dado)
periods_raw = pd.read_csv(PATH_PERIODS, header=None)
periods_raw.columns = ["dias_sem", "nan1", "nan2", "dias_ano"]
periods_raw = periods_raw[["dias_sem", "dias_ano"]]

print(f"Semestres encontrados: {len(periods_raw)}")
print(f"Total de dias de pregão: {periods_raw['dias_sem'].sum():,}")
print()
print(periods_raw)


Semestres encontrados: 51
Total de dias de pregão: 6,425

    dias_sem  dias_ano
0        127     252.0
1        125     253.0
2        128     254.0
3        126     254.0
4        128     252.0
5        124     252.0
6        128     253.0
7        125     252.0
8        127     253.0
9        126     252.0
10       126     252.0
11       126     254.0
12       128     253.0
13       125     253.0
14       128     252.0
15       124     252.0
16       128     252.0
17       124     252.0
18       128     254.0
19       126     252.0
20       126     251.0
21       125     248.0
22       123     247.0
23       124     252.0
24       128     252.0
25       124     252.0
26       128     252.0
27       124     252.0
28       128     253.0
29       125     252.0
30       127     252.0
31       125     251.0
32       126     250.0
33       124     251.0
34       127     252.0
35       125     253.0
36       128     252.0
37       124     252.0
38       128     252.0
39       124     252.0

In [85]:
# Construir mapeamento linha → semestre usando Periods.csv como fonte verdadeira.
# Regra de indexação:
#   idx 0          → 1990/S2 (caso especial)
#   idx ímpar ≥ 1  → S1 do ano (1990 + idx // 2)
#   idx par   ≥ 2  → S2 do ano (1990 + idx // 2)

registros = []
linha_atual = 0

for i, row in periods_raw.iterrows():
    n_dias = int(row["dias_sem"])

    if i == 0:
        ano, sem = 1990, 2
        inicio_sem, fim_sem = "1990-07-01", "1990-12-31"
    else:
        ano = 1990 + (i // 2)
        if i % 2 == 1:      # ímpar → primeiro semestre
            sem = 1
            inicio_sem = f"{ano}-01-01"
            fim_sem    = f"{ano}-06-30"
        else:               # par → segundo semestre
            sem = 2
            inicio_sem = f"{ano}-07-01"
            fim_sem    = f"{ano}-12-31"

    for d in range(n_dias):
        registros.append({
            "linha":           linha_atual,
            "ano":             ano,
            "semestre":        sem,
            "periodo":         f"{ano}/S{sem}",
            "inicio_semestre": inicio_sem,
            "fim_semestre":    fim_sem,
            "dia_no_semestre": d + 1,
            "total_dias_sem":  n_dias,
            "total_dias_ano":  row["dias_ano"],
        })
        linha_atual += 1

mapa = pd.DataFrame(registros)
print(f"Linhas mapeadas: {len(mapa):,}")
print()
print("Início:")
print(mapa.head(3)[["linha", "periodo", "dia_no_semestre", "total_dias_sem"]])
print("\nFim:")
print(mapa.tail(3)[["linha", "periodo", "dia_no_semestre", "total_dias_sem"]])


Linhas mapeadas: 6,425

Início:
   linha  periodo  dia_no_semestre  total_dias_sem
0      0  1990/S2                1             127
1      1  1990/S2                2             127
2      2  1990/S2                3             127

Fim:
      linha  periodo  dia_no_semestre  total_dias_sem
6422   6422  2015/S2              125             127
6423   6423  2015/S2              126             127
6424   6424  2015/S2              127             127


In [86]:
# ── Validações do mapeamento ──────────────────────────────────────────────────

# Validação 1: 2001/S2 deve ter exatamente 123 dias (11 de setembro)
dias_2001s2 = len(mapa[mapa["periodo"] == "2001/S2"])
ok1 = dias_2001s2 == 123
print(f"{'✅' if ok1 else '❌'} 2001/S2 tem {dias_2001s2} dias (esperado: 123 — efeito 11/set)")

# Validação 2: GOOGL deve aparecer em 2004/S2 (IPO agosto/2004)
if "GOOGL" in db.columns:
    idx_googl = db["GOOGL"].first_valid_index()
    periodo_googl = mapa.iloc[idx_googl]["periodo"]
    dia_googl = mapa.iloc[idx_googl]["dia_no_semestre"]
    ok2 = periodo_googl == "2004/S2"
    print(f"{'✅' if ok2 else '❌'} GOOGL aparece na linha {idx_googl} → {periodo_googl}, "
          f"dia {dia_googl} do semestre (esperado: 2004/S2)")

# Validação 3: diferença de linhas entre db e mapa
diff = abs(len(db) - len(mapa))
ok3 = diff <= 1
print(f"{'✅' if ok3 else '⚠️ '} db tem {len(db):,} linhas, mapa tem {len(mapa):,} "
      f"(diferença: {diff} — ok se ≤ 1)")


✅ 2001/S2 tem 123 dias (esperado: 123 — efeito 11/set)
✅ GOOGL aparece na linha 3563 → 2004/S2, dia 35 do semestre (esperado: 2004/S2)
✅ db tem 6,426 linhas, mapa tem 6,425 (diferença: 1 — ok se ≤ 1)


---
## Etapa 4 — Unir base de preços com metadados de período

Adiciona 8 colunas de contexto temporal no início da base:
`ano`, `semestre`, `periodo`, `inicio_semestre`, `fim_semestre`,
`dia_no_semestre`, `total_dias_sem`, `total_dias_ano`.

O resultado é salvo em `sp500_with_periods.csv`.


In [87]:
# Ajustar tamanho do mapa para coincidir com db (diferença de ±1 é normal)
if len(mapa) > len(db):
    mapa_alinhado = mapa.iloc[:len(db)].reset_index(drop=True)
elif len(mapa) < len(db):
    extras = pd.DataFrame([{
        "linha": len(mapa) + k, "ano": np.nan, "semestre": np.nan,
        "periodo": np.nan, "inicio_semestre": np.nan, "fim_semestre": np.nan,
        "dia_no_semestre": np.nan, "total_dias_sem": np.nan, "total_dias_ano": np.nan,
    } for k in range(len(db) - len(mapa))])
    mapa_alinhado = pd.concat([mapa, extras], ignore_index=True)
else:
    mapa_alinhado = mapa.reset_index(drop=True)

cols_periodo = [
    "ano", "semestre", "periodo",
    "inicio_semestre", "fim_semestre",
    "dia_no_semestre", "total_dias_sem", "total_dias_ano",
]

sp500_with_periods = pd.concat(
    [mapa_alinhado[cols_periodo], db.reset_index(drop=True)],
    axis=1
)

print(f"Shape final: {sp500_with_periods.shape[0]:,} linhas × {sp500_with_periods.shape[1]:,} colunas")
print(f"Primeiro período: {sp500_with_periods['periodo'].iloc[0]}")
print(f"Último  período: {sp500_with_periods['periodo'].dropna().iloc[-1]}")


Shape final: 6,426 linhas × 1,108 colunas
Primeiro período: 1990/S2
Último  período: 2015/S2


In [88]:
# Salvar a base unificada
sp500_with_periods.to_csv(PATH_DB_PERIODS, index=False)
print(f"✅ Salvo em: {PATH_DB_PERIODS.name}")
print()

# Amostra para inspeção visual
sp500_with_periods[["periodo", "ano", "semestre", "dia_no_semestre", "AAPL", "GE", "XOM"]].head(5)


✅ Salvo em: sp500_with_periods.csv



,periodo,ano,semestre,dia_no_semestre,AAPL,GE,XOM
0,1990/S2,1990.0,2.0,1.0,1.3753,2.8747,5.6624
1,1990/S2,1990.0,2.0,2.0,1.3753,2.9053,5.6328
2,1990/S2,1990.0,2.0,3.0,1.3596,2.8645,5.5438
3,1990/S2,1990.0,2.0,4.0,1.3987,2.8798,5.6624
4,1990/S2,1990.0,2.0,5.0,1.4573,2.9104,5.6180


---
## Etapa 5 — Constituintes históricos do S&P 500 (1996–2026)

O arquivo `S&P 500 Historical Components & Changes` (repositório `fja05680/sp500`)
registra cada mudança de composição do índice desde 1996. Cada linha representa
uma data em que a composição mudou, com a lista completa de tickers naquele momento.

Para saber quem estava no índice em qualquer data D: buscamos a última linha
com `date ≤ D`.

**Validações usadas:**
- Tesla (`TSLA`): entrou em **21/dez/2020** (não antes)
- Facebook/Meta: era `FB` até **08/jun/2022**, virou `META` em 09/jun/2022
- Lehman Brothers (`LEHMQ`): removida após a falência em setembro de 2008


In [89]:
hist = pd.read_csv(PATH_SP500HIS)
hist["date"] = pd.to_datetime(hist["date"])
hist["n_tickers"] = hist["tickers"].apply(lambda x: len(x.split(",")))

print(f"Snapshots carregados: {len(hist):,}")
print(f"Período coberto: {hist['date'].min().date()} → {hist['date'].max().date()}")
print(f"Tickers por snapshot: min={hist['n_tickers'].min()}, max={hist['n_tickers'].max()}, "
      f"média={hist['n_tickers'].mean():.0f}")


Snapshots carregados: 2,705
Período coberto: 1996-01-02 → 2026-01-14
Tickers por snapshot: min=487, max=507, média=497


In [90]:
def get_constituents(date_str):
    """Retorna o conjunto de tickers do S&P 500 em uma data específica."""
    date = pd.to_datetime(date_str)
    subset = hist[hist["date"] <= date]
    if len(subset) == 0:
        return set()
    return set(subset.iloc[-1]["tickers"].split(","))

# ── Validações cruzadas ───────────────────────────────────────────────────────
validations = [
    ("TSLA",  True,  "2020-12-21", "Tesla entrou em 21/dez/2020"),
    ("TSLA",  False, "2020-12-20", "Tesla NÃO estava em 20/dez/2020"),
    ("FB",    True,  "2022-06-08", "Facebook ainda era FB em 08/jun/2022"),
    ("META",  True,  "2022-06-09", "Meta a partir de 09/jun/2022"),
    ("LEHMQ", False, "2009-01-01", "Lehman removida após falência em 2008"),
    ("AMZN",  True,  "2023-12-31", "Amazon sempre presente"),
]

all_ok = True
for ticker, should_be_in, date_str, note in validations:
    tickers = get_constituents(date_str)
    present = ticker in tickers
    ok = present == should_be_in
    if not ok:
        all_ok = False
    status = "✅" if ok else "❌"
    print(f"  {status} {ticker:6s} {'IN ' if should_be_in else 'OUT'}  {date_str}  — {note}")

print()
print("Todas as validações passaram ✅" if all_ok else "⚠️  Verificar falhas acima.")


  ✅ TSLA   IN   2020-12-21  — Tesla entrou em 21/dez/2020
  ✅ TSLA   OUT  2020-12-20  — Tesla NÃO estava em 20/dez/2020
  ✅ FB     IN   2022-06-08  — Facebook ainda era FB em 08/jun/2022
  ✅ META   IN   2022-06-09  — Meta a partir de 09/jun/2022
  ✅ LEHMQ  OUT  2009-01-01  — Lehman removida após falência em 2008
  ✅ AMZN   IN   2023-12-31  — Amazon sempre presente

Todas as validações passaram ✅


---
## Etapa 6 — Snapshots semestrais e lista de tickers para coleta

Para cada semestre de **2015/S1** a **2025/S2**, extraímos a composição do
índice no último dia do semestre (30/jun ou 31/dez). Isso define quem estava
"dentro" do índice naquele período — respeitando o critério anti-survivorship bias.

O resultado são dois arquivos:
- `snapshots_semestrais.csv` — composição completa por semestre
- `tickers_a_coletar.csv` — lista de tickers únicos com janela de coleta no Yahoo


In [91]:
# Definir os semestres de 2015/S1 a 2025/S2
semesters = []
for ano in range(2015, 2026):
    for sem in [1, 2]:
        inicio = f"{ano}-01-01" if sem == 1 else f"{ano}-07-01"
        fim    = f"{ano}-06-30" if sem == 1 else f"{ano}-12-31"
        semesters.append({
            "periodo":       f"{ano}/S{sem}",
            "ano":           ano,
            "semestre":      sem,
            "inicio":        inicio,
            "fim":           fim,
            "snapshot_date": fim,   # composição no último dia do semestre
        })

# Gerar snapshots
snap_rows = []
for s in semesters:
    t = get_constituents(s["snapshot_date"])
    snap_rows.append({**s, "n_tickers": len(t), "tickers": ",".join(sorted(t))})

snap_df = pd.DataFrame(snap_rows)
snap_df.to_csv(PATH_SNAPSHOTS, index=False)

print(f"Snapshots gerados: {len(snap_df)} semestres")
print()
print(snap_df[["periodo", "n_tickers"]].to_string(index=False))


Snapshots gerados: 22 semestres

periodo  n_tickers
2015/S1        499
2015/S2        502
2016/S1        505
2016/S2        506
2017/S1        506
2017/S2        505
2018/S1        506
2018/S2        505
2019/S1        505
2019/S2        505
2020/S1        505
2020/S2        505
2021/S1        505
2021/S2        505
2022/S1        503
2022/S2        503
2023/S1        503
2023/S2        503
2024/S1        503
2024/S2        503
2025/S1        503
2025/S2        503


In [92]:
# Construir lista de todos os tickers únicos no período de extensão (excluindo 2015/S1,
# que já está na base do professor).
ext_df = snap_df[snap_df["periodo"] != "2015/S1"].copy()

all_tickers = set()
for _, row in ext_df.iterrows():
    all_tickers.update(row["tickers"].split(","))

# Para cada ticker: janela de coleta (primeiro e último semestre em que esteve no índice)
ticker_rows = []
for ticker in sorted(all_tickers):
    periods_in = [
        row["periodo"]
        for _, row in ext_df.iterrows()
        if ticker in row["tickers"].split(",")
    ]
    first_sem, last_sem = periods_in[0], periods_in[-1]
    f_ano, f_s = int(first_sem[:4]), int(first_sem[-1])
    l_ano, l_s = int(last_sem[:4]),  int(last_sem[-1])
    start_date = f"{f_ano}-01-01" if f_s == 1 else f"{f_ano}-07-01"
    end_date   = f"{l_ano}-06-30" if l_s == 1 else f"{l_ano}-12-31"

    ticker_rows.append({
        "ticker":       ticker,
        "first_period": first_sem,
        "last_period":  last_sem,
        "n_periods":    len(periods_in),
        "start_date":   start_date,
        "end_date":     end_date,
        "status":       "pending",
    })

tickers_df = pd.DataFrame(ticker_rows)
tickers_df.to_csv(PATH_TICKERS, index=False)

# Resumo de turnover
t_2015s1 = set(snap_df[snap_df["periodo"] == "2015/S1"]["tickers"].iloc[0].split(","))
t_2025s2 = set(snap_df[snap_df["periodo"] == "2025/S2"]["tickers"].iloc[0].split(","))
print(f"Tickers únicos para coletar: {len(tickers_df)}")
print(f"  Permaneceram (2015→2025):  {len(t_2015s1 & t_2025s2)}")
print(f"  Entraram no índice:        {len(t_2025s2 - t_2015s1)}")
print(f"  Saíram do índice:          {len(t_2015s1 - t_2025s2)}")
print()
print(f"Arquivo salvo: {PATH_TICKERS.name}")


Tickers únicos para coletar: 725
  Permaneceram (2015→2025):  320
  Entraram no índice:        183
  Saíram do índice:          179

Arquivo salvo: tickers_a_coletar.csv


---
## Etapa 7 — Validação do tipo de ajuste de preço

Antes de coletar os dados do Yahoo, precisamos confirmar qual tipo de preço
a base do professor usa — para garantir consistência na junção das bases.

### O que descobrimos

A base do professor usa **`Close` sem ajuste de dividendos** (split-adjusted apenas),
**não** o `Adj Close` que inclui dividendos.

**Evidência:** ao comparar os últimos valores da base do professor com o
Yahoo Finance no mesmo período, o `ratio` com `Close` bruto fica ≈ 1.0
para MSFT, KO, JNJ e XOM, enquanto o `ratio` com `Adj Close` diverge
significativamente (1.15×, 1.38×, 1.33×, 1.56×).

### Por que a validação exige a base completa

O script usa `Periods.csv` para mapear linha → data exata, evitando o
desalinhamento causado por feriados da NYSE. A base parcial (4k linhas)
usa `bdate_range` como proxy e acumula ~2-3 dias de erro até o final —
suficiente para inverter a correlação dos retornos diários e gerar falsos
negativos na validação.

### Consequência para a coleta (Etapa 8)

Usar `auto_adjust=False` e a coluna `Close` no yfinance — não `Adj Close`.


In [93]:
db_full

,UN,1005945D,162007Q,RDPL,BGG,SKY,GOSHA,HSY,NMK,WLB,...,MNST,O,JBHT,BXLT,ENDP,EQIX,1436513D,1431816D,SPGI,1448062D
1990-07-02,3.3082,11.7377,32.6951,10.2566,3.6022,7.723,13.6570,5.3707,10.9995,23.0656,...,NaN,NaN,2.8041,NaN,NaN,NaN,1.4481,2.7908,3.875790,NaN
1990-07-03,3.3321,11.8106,32.1859,10.2396,3.5747,7.657,13.7545,5.3175,10.9022,23.0656,...,NaN,NaN,2.7715,NaN,NaN,NaN,1.4655,2.7288,3.858865,NaN
1990-07-04,3.2699,11.5919,30.7599,10.1207,3.4922,7.591,13.4619,5.2112,10.7075,23.0656,...,NaN,NaN,2.7715,NaN,NaN,NaN,1.4655,2.6668,3.841940,NaN
1990-07-05,3.3082,11.6648,32.0840,10.1886,3.4647,7.657,13.6570,5.3175,10.8048,23.0656,...,NaN,NaN,2.7715,NaN,NaN,NaN,1.4713,2.7288,3.841940,NaN
1990-07-06,3.2986,11.6648,31.6766,10.2226,3.4510,7.591,13.6570,5.3884,10.8048,22.8278,...,NaN,NaN,2.6737,NaN,NaN,NaN,1.4365,2.6357,3.841940,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-12-21,43.6300,NaN,NaN,NaN,17.0200,NaN,NaN,90.3400,NaN,6.4700,...,147.47,51.1365,72.6700,39.33,62.71,297.35,39.8606,NaN,94.632677,NaN
2015-12-22,43.5600,NaN,NaN,NaN,16.9300,NaN,NaN,90.3200,NaN,6.3800,...,148.82,50.9280,72.9700,39.27,62.62,297.19,39.6647,NaN,96.685191,NaN
2015-12-23,43.7100,NaN,NaN,NaN,16.7100,NaN,NaN,90.3900,NaN,5.9900,...,148.16,51.3748,72.8800,38.49,61.82,300.91,39.0673,NaN,96.201661,NaN
2015-12-24,44.2300,NaN,NaN,NaN,17.8500,NaN,NaN,90.9800,NaN,5.8900,...,150.17,51.8016,73.5100,39.39,62.49,304.98,39.6843,NaN,96.181929,NaN


In [94]:
# ── Validação do ajuste de preço ──────────────────────────────────────────────
# Compara os últimos valores da base do professor com o Yahoo Finance,
# usando as datas reais extraídas do Periods.csv (sem feriados incorretos).
#
# Requer a base COMPLETA (6426 linhas). Se ainda não tiver o arquivo completo,
# comente este bloco e prossiga — a conclusão já é conhecida: usar Close bruto.

TICKERS_VALIDACAO = ["MSFT", "KO", "JNJ", "XOM"]
N_DIAS_COMP       = 20    # pregões finais da base para comparar
TOLERANCIA_RATIO  = 0.01  # diferença máxima aceitável no ratio (1%)

if not PATH_DB_FULL.exists():
    print("⚠️  Base completa não encontrada. Pulando validação.")
    print("   Conclusão conhecida: usar auto_adjust=False, coluna 'Close'.")
else:
    # Carregar base completa e reconstruir mapa de datas via Periods.csv
    print("Carregando base completa e mapa de períodos...")
    db_full = pd.read_csv(PATH_DB_FULL)

    periods_v = pd.read_csv(PATH_PERIODS, header=None)
    periods_v.columns = ["dias_sem", "nan1", "nan2", "dias_ano"]

    # Reconstruir datas usando pd.bdate_range corrigido por semestre
    # (cada semestre tem início certo; erro de feriados fica contido ao semestre)
    datas = []
    for i, row in periods_v.iterrows():
        n = int(row["dias_sem"])
        if i == 0:   ini = "1990-07-02"
        else:
            ano = 1990 + (i // 2)
            ini = f"{ano}-01-02" if i % 2 == 1 else f"{ano}-07-02"
        datas.extend(pd.bdate_range(start=ini, periods=n))

    n_alinhar = min(len(db_full), len(datas))
    db_full = db_full.iloc[:n_alinhar]  # ← add this line
    db_full.index = pd.DatetimeIndex(datas[:n_alinhar])

    print(f"  Base: {len(db_full):,} linhas | última data mapeada: {db_full.index[-1].date()}")
    print()

    resultados = []
    for ticker in TICKERS_VALIDACAO:
        if ticker not in db_full.columns:
            print(f"{ticker}: não encontrado na base"); continue

        serie_prof = db_full[ticker].dropna().tail(N_DIAS_COMP)
        if len(serie_prof) < 5:
            print(f"{ticker}: dados insuficientes"); continue

        d_ini = serie_prof.index[0].strftime("%Y-%m-%d")
        d_fim = (serie_prof.index[-1] + pd.Timedelta(days=5)).strftime("%Y-%m-%d")

        try:
            raw = yf.download(ticker, start=d_ini, end=d_fim,
                              auto_adjust=False, progress=False)
            if hasattr(raw.columns, "levels"):
                raw.columns = raw.columns.get_level_values(0)
            if raw is None or len(raw) == 0:
                print(f"{ticker}: sem dados no Yahoo"); continue

            idx_comum = serie_prof.index.intersection(raw.index)
            if len(idx_comum) < 3:
                print(f"{ticker}: apenas {len(idx_comum)} datas em comum"); continue

            prof     = serie_prof.loc[idx_comum]
            y_close  = raw["Close"].loc[idx_comum]
            y_adj    = raw["Adj Close"].loc[idx_comum]

            ratio_close = (prof / y_close).replace([float("inf"), float("-inf")], float("nan")).dropna()
            ratio_adj   = (prof / y_adj).replace([float("inf"), float("-inf")], float("nan")).dropna()

            dist_close = abs(ratio_close.mean() - 1.0)
            dist_adj   = abs(ratio_adj.mean()   - 1.0)
            tipo = "Close (sem div)" if dist_close < dist_adj else "Adj Close (com div)"

            icon = "✅" if dist_close < TOLERANCIA_RATIO else "⚠️ "
            print(f"{icon} {ticker:6s} | ratio Close={ratio_close.mean():.4f} (±{ratio_close.std():.4f})"
                  f" | ratio AdjClose={ratio_adj.mean():.4f} (±{ratio_adj.std():.4f})"
                  f" | mais próximo: {tipo}")

            resultados.append({"ticker": ticker, "tipo": tipo,
                                "ratio_close": ratio_close.mean(),
                                "ratio_adj":   ratio_adj.mean()})

        except Exception as e:
            print(f"{ticker}: ERRO — {e}")

    print()
    if resultados:
        tipos = [r["tipo"] for r in resultados]
        dominante = max(set(tipos), key=tipos.count)
        if "Close" in dominante and "sem div" in dominante:
            print("✅ CONCLUSÃO: base do professor usa Close SEM ajuste de dividendos.")
            print("   → Na coleta (Etapa 8): usar auto_adjust=False, coluna 'Close'.")
        else:
            print("⚠️  CONCLUSÃO: base parece usar Adj Close (com dividendos).")
            print("   → Na coleta (Etapa 8): usar auto_adjust=True, coluna 'Close'.")
    else:
        print("Não foi possível concluir — verificar manualmente.")


Carregando base completa e mapa de períodos...
  Base: 6,425 linhas | última data mapeada: 2015-12-25

✅ MSFT   | ratio Close=1.0029 (±0.0176) | ratio AdjClose=1.1480 (±0.0201) | mais próximo: Close (sem div)
✅ KO     | ratio Close=1.0022 (±0.0157) | ratio AdjClose=1.3817 (±0.0216) | mais próximo: Close (sem div)
✅ JNJ    | ratio Close=1.0020 (±0.0159) | ratio AdjClose=1.3287 (±0.0211) | mais próximo: Close (sem div)
✅ XOM    | ratio Close=0.9967 (±0.0298) | ratio AdjClose=1.5549 (±0.0465) | mais próximo: Close (sem div)

✅ CONCLUSÃO: base do professor usa Close SEM ajuste de dividendos.
   → Na coleta (Etapa 8): usar auto_adjust=False, coluna 'Close'.


---
## Etapa 8 — Coleta de preços via Yahoo Finance

Para cada ticker em `tickers_a_coletar.csv`, baixamos o preço de fechamento
**sem ajuste de dividendos** (`auto_adjust=False`, coluna `Close`) — consistente
com a metodologia da base do professor.

> **Por que não usar `Adj Close`?**
> O `Adj Close` do Yahoo desconta dividendos retroativamente toda vez que um
> novo dividendo é pago, alterando valores históricos a cada atualização.
> A base do professor usa apenas ajuste por splits — usar `Adj Close` criaria
> uma descontinuidade artificial na junção das bases em 2015/S1.

**Checkpoint de retomada:** o `coleta_report.csv` é salvo após cada ticker.
Se a coleta for interrompida, basta re-executar — os tickers já coletados
são pulados automaticamente.

**Tickers `DELISTED`:** empresas que faliram ou foram adquiridas não têm
histórico no Yahoo. Elas serão listadas no sumário final para busca em
fontes alternativas (Tiingo ou EODHD).


In [95]:
# ── Configurações da coleta ───────────────────────────────────────────────────
DELAY_ENTRE_REQUESTS = 0.3   # segundos entre requests (evita rate limit do Yahoo)
MIN_LINHAS_VALIDAS   = 10    # mínimo de dias para considerar o download válido

# Recarregar lista para garantir que está atualizada
tickers_df = pd.read_csv(PATH_TICKERS)

# Carregar relatório existente (checkpoint de retomada)
if PATH_REPORT.exists():
    report_df    = pd.read_csv(PATH_REPORT)
    already_done = set(report_df["ticker"].tolist())
    print(f"Retomando: {len(already_done)} já coletados, "
          f"{len(tickers_df) - len(already_done)} restantes.")
else:
    report_df    = pd.DataFrame(columns=["ticker", "status", "n_rows",
                                          "start_date", "end_date", "note"])
    already_done = set()
    print(f"Iniciando coleta de {len(tickers_df)} tickers.")

print(f"Ajuste de preço: Close sem dividendos (auto_adjust=False)")


Iniciando coleta de 725 tickers.
Ajuste de preço: Close sem dividendos (auto_adjust=False)


In [96]:
# ── Loop de coleta ────────────────────────────────────────────────────────────
total = len(tickers_df)

for _, row in tickers_df.iterrows():
    ticker     = row["ticker"]
    start_date = row["start_date"]
    end_date   = row["end_date"]

    if ticker in already_done:
        continue

    n_done = len(already_done) + 1
    print(f"[{n_done:4d}/{total}] {ticker:10s}  {start_date} → {end_date}  ", end="", flush=True)

    try:
        # auto_adjust=False → coluna 'Close' é split-adjusted apenas (sem dividendos)
        # Isso é consistente com a metodologia da base do professor.
        data = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False,
        )

        # Normalizar colunas (yfinance pode retornar MultiIndex)
        if hasattr(data.columns, "levels"):
            data.columns = data.columns.get_level_values(0)

        if data is None or len(data) < MIN_LINHAS_VALIDAS:
            status, note, n_rows = "delisted", "sem dados no Yahoo", 0
            print("DELISTED")
        else:
            # Salvar apenas a coluna Close (sem ajuste de dividendos)
            price_series = data[["Close"]].rename(columns={"Close": ticker})
            price_series.to_csv(DIR_PRICES / f"{ticker}.csv")
            status, note, n_rows = "ok", "", len(data)
            print(f"OK  ({n_rows} dias)")

    except Exception as e:
        status, note, n_rows = "error", str(e)[:120], 0
        print(f"ERRO: {note}")

    # Salvar no relatório imediatamente (garante retomada mesmo com crash)
    new_row = pd.DataFrame([{
        "ticker":     ticker,
        "status":     status,
        "n_rows":     n_rows,
        "start_date": start_date,
        "end_date":   end_date,
        "note":       note,
    }])
    report_df    = pd.concat([report_df, new_row], ignore_index=True)
    report_df.to_csv(PATH_REPORT, index=False)
    already_done.add(ticker)

    time.sleep(DELAY_ENTRE_REQUESTS)

print("\nColeta concluída.")


[   1/725] A           2015-07-01 → 2025-12-31  OK  (2641 dias)
[   2/725] AABA        2015-07-01 → 2016-12-31  

$AABA: possibly delisted; no timezone found

1 Failed download:
['AABA']: possibly delisted; no timezone found


DELISTED
[   3/725] AAL         2015-07-01 → 2024-06-30  OK  (2264 dias)
[   4/725] AAP         2015-07-01 → 2023-06-30  OK  (2013 dias)
[   5/725] AAPL        2015-07-01 → 2025-12-31  OK  (2641 dias)
[   6/725] ABBV        2015-07-01 → 2025-12-31  OK  (2641 dias)
[   7/725] ABC         2015-07-01 → 2023-06-30  

$ABC: possibly delisted; no timezone found

1 Failed download:
['ABC']: possibly delisted; no timezone found


DELISTED
[   8/725] ABMD        2018-01-01 → 2022-06-30  

...

DELISTED
[ 713/725] XEL         2015-07-01 → 2025-12-31  OK  (2641 dias)
[ 714/725] XL          2015-07-01 → 2018-06-30  

$XL: possibly delisted; no timezone found

1 Failed download:
['XL']: possibly delisted; no timezone found


DELISTED
[ 715/725] XLNX        2015-07-01 → 2021-12-31  

$XLNX: possibly delisted; no timezone found

1 Failed download:
['XLNX']: possibly delisted; no timezone found


DELISTED
[ 716/725] XOM         2015-07-01 → 2025-12-31  OK  (2641 dias)
[ 717/725] XRAY        2015-07-01 → 2023-12-31  OK  (2140 dias)
[ 718/725] XRX         2015-07-01 → 2020-12-31  OK  (1386 dias)
[ 719/725] XYL         2015-07-01 → 2025-12-31  OK  (2641 dias)
[ 720/725] XYZ         2025-07-01 → 2025-12-31  OK  (127 dias)
[ 721/725] YUM         2015-07-01 → 2025-12-31  OK  (2641 dias)
[ 722/725] ZBH         2015-07-01 → 2025-12-31  OK  (2641 dias)
[ 723/725] ZBRA        2019-07-01 → 2025-12-31  OK  (1635 dias)
[ 724/725] ZION        2015-07-01 → 2023-12-31  OK  (2140 dias)
[ 725/725] ZTS         2015-07-01 → 2025-12-31  OK  (2641 dias)

Coleta concluída.


In [98]:
# ── Sumário final da coleta ───────────────────────────────────────────────────
report_df = pd.read_csv(PATH_REPORT)

contagem = report_df["status"].value_counts()
print("=== SUMÁRIO DA COLETA ===")
print(f"  ✅ OK (dados baixados):    {contagem.get('ok', 0)}")
print(f"  ⚠️  Delisted/sem dados:   {contagem.get('delisted', 0)}")
print(f"  ❌ Erros:                 {contagem.get('error', 0)}")
print(f"  Total:                    {len(report_df)}")
print()

# Listar tickers delisted para busca em fontes alternativas (Tiingo / EODHD)
delisted = report_df[report_df["status"] == "delisted"]
if len(delisted) > 0:
    tickers_df = pd.read_csv(PATH_TICKERS)
    print(f"Tickers delisted ({len(delisted)}) — buscar no Tiingo ou EODHD:")
    for _, r in delisted.iterrows():
        t_info = tickers_df[tickers_df["ticker"] == r["ticker"]]
        if len(t_info) > 0:
            t = t_info.iloc[0]
            print(f"  {r['ticker']:10s}  {t['first_period']} → {t['last_period']}")


=== SUMÁRIO DA COLETA ===
  ✅ OK (dados baixados):    592
  ⚠️  Delisted/sem dados:   133
  ❌ Erros:                 0
  Total:                    725

Tickers delisted (133) — buscar no Tiingo ou EODHD:
  AABA        2015/S2 → 2016/S2
  ABC         2015/S2 → 2023/S1
  ABMD        2018/S1 → 2022/S1
  ADS         2015/S2 → 2019/S2
  ADT         2015/S2 → 2015/S2
  AGN         2015/S2 → 2019/S2
  ALXN        2015/S2 → 2021/S1
  ANSS        2017/S1 → 2025/S1
  ANTM        2015/S2 → 2021/S2
  APC         2015/S2 → 2019/S1
  ARG         2015/S2 → 2015/S2
  ARNC        2015/S2 → 2019/S2
  ATVI        2015/S2 → 2023/S1
  BCR         2015/S2 → 2017/S1
  BF.B        2015/S2 → 2025/S2
  BHGE        2015/S2 → 2019/S1
  BLL         2015/S2 → 2021/S2
  BRCM        2015/S2 → 2015/S2
  BRK.B       2015/S2 → 2025/S2
  BXLT        2015/S2 → 2015/S2
  CA          2015/S2 → 2018/S1
  CAM         2015/S2 → 2015/S2
  CBS         2015/S2 → 2019/S1
  CCE         2015/S2 → 2015/S2
  CDAY        2021/S2 → 2023

In [99]:
delisted

,ticker,status,n_rows,start_date,end_date,note
1,AABA,delisted,0,2015-07-01,2016-12-31,sem dados no Yahoo
6,ABC,delisted,0,2015-07-01,2023-06-30,sem dados no Yahoo
7,ABMD,delisted,0,2018-01-01,2022-06-30,sem dados no Yahoo
16,ADS,delisted,0,2015-07-01,2019-12-31,sem dados no Yahoo
18,ADT,delisted,0,2015-07-01,2015-12-31,sem dados no Yahoo
...,...,...,...,...,...,...
703,WRK,delisted,0,2015-07-01,2024-06-30,sem dados no Yahoo
709,WYND,delisted,0,2015-07-01,2017-12-31,sem dados no Yahoo
711,XEC,delisted,0,2015-07-01,2019-12-31,sem dados no Yahoo
713,XL,delisted,0,2015-07-01,2018-06-30,sem dados no Yahoo


In [100]:
report_df

,ticker,status,n_rows,start_date,end_date,note
0,A,ok,2641,2015-07-01,2025-12-31,NaN
1,AABA,delisted,0,2015-07-01,2016-12-31,sem dados no Yahoo
2,AAL,ok,2264,2015-07-01,2024-06-30,NaN
3,AAP,ok,2013,2015-07-01,2023-06-30,NaN
4,AAPL,ok,2641,2015-07-01,2025-12-31,NaN
...,...,...,...,...,...,...
720,YUM,ok,2641,2015-07-01,2025-12-31,NaN
721,ZBH,ok,2641,2015-07-01,2025-12-31,NaN
722,ZBRA,ok,1635,2019-07-01,2025-12-31,NaN
723,ZION,ok,2140,2015-07-01,2023-12-31,NaN


In [ ]:
#tiingo API token: dadfd331f2cb44969b8f7468006d20ad62b13262

Plano de recuperação dos 133 delisted
Os 133 se dividem em quatro grupos com estratégias diferentes:
Tipo 1 — Problema de formato (2 tickers) → Yahoo, só mudar o ticker
BF.B e BRK.B usam ponto, mas o Yahoo usa hífen: BF-B e BRK-B. Resolve com um retry simples.
Tipo 2 — Renomeação de ticker (10 tickers) → Yahoo com novo ticker + concatenar
A empresa existe, só o ticker mudou. Ex: FB → META, ANTM → ELV, CTL → LUMN. Busca no Yahoo com o novo nome e junta o histórico antigo com o novo.
Tipo 3 — Adquiridas/fundidas (37 tickers) → Tiingo
O ticker sumiu do Yahoo porque a empresa foi absorvida. O Tiingo mantém histórico de tickers delistados. Ex: CELG (adquirida BMS), XLNX (adquirida AMD), TWTR (privatizada).
Tipo 4 — Falências reais (7 tickers) → Tiingo
SIVB, FRC, SBNY (bancos de 2023), ENDP, MNK etc. O Tiingo tem o histórico até a data da falência, que é exatamente o que precisamos.
Não mapeados (77 tickers) → Tiingo como fallback geral para todos que o Yahoo não entrega.

In [102]:
import requests
headers = {
    'Content-Type': 'application/json'
}
requestResponse = requests.get("https://api.tiingo.com/tiingo/daily/aaba/prices?startDate=2015-07-02&token=dadfd331f2cb44969b8f7468006d20ad62b13262&resampleFreq=daily", headers=headers)
print(requestResponse.json())

[{'date': '2015-07-02T00:00:00.000Z', 'close': 39.38, 'high': 39.64, 'low': 39.19, 'open': 39.26, 'volume': 7712994, 'adjClose': 10.8196563864, 'adjHigh': 10.8910913956, 'adjLow': 10.7674538797, 'adjOpen': 10.7866863822, 'adjVolume': 7712994, 'divCash': 0.0, 'splitFactor': 1.0}, {'date': '2015-07-06T00:00:00.000Z', 'close': 38.61, 'high': 39.12, 'low': 38.46, 'open': 38.76, 'volume': 11803417, 'adjClose': 10.6080988593, 'adjHigh': 10.7482213773, 'adjLow': 10.566886354, 'adjOpen': 10.6493113646, 'adjVolume': 11803417, 'divCash': 0.0, 'splitFactor': 1.0}, {'date': '2015-07-07T00:00:00.000Z', 'close': 38.23, 'high': 38.38, 'low': 36.575, 'open': 38.24, 'volume': 19432474, 'adjClose': 10.5036938459, 'adjHigh': 10.5449063512, 'adjLow': 10.0489825377, 'adjOpen': 10.5064413463, 'adjVolume': 19432474, 'divCash': 0.0, 'splitFactor': 1.0}, {'date': '2015-07-08T00:00:00.000Z', 'close': 37.23, 'high': 37.49, 'low': 36.94, 'open': 37.2, 'volume': 20529176, 'adjClose': 10.2289438107, 'adjHigh': 10.3

In [104]:
def load_yahoo_price(ticker):
    path = Path(DIR_PRICES) / f"{ticker}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Yahoo price file missing: {path}")
    df = pd.read_csv(path, parse_dates=[0])
    df.rename(columns={df.columns[0]: "date"}, inplace=True)
    df.set_index("date", inplace=True)
    price_cols = [c for c in df.columns if c.lower() in {"close", ticker.lower()}]
    if not price_cols:
        raise ValueError(f"Yahoo file for {ticker} has no Close-like column")
    df = df.rename(columns={price_cols[0]: "Close"})
    return df


In [108]:
import requests
import yfinance as yf


def fetch_tiingo(ticker, start_date, end_date):
    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"
    params = {
        'startDate': start_date,
        'endDate': end_date,
        'token': TIINGO_TOKEN,
        'resampleFreq': 'daily'
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        raise ValueError(f"Tiingo API error for {ticker}: {response.status_code}")
    data = response.json()
    df = pd.DataFrame(data)
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    df.index = df.index.tz_localize(None)
    if 'close' not in df.columns:
        raise ValueError(f"Tiingo response for {ticker} has no close column")
    return df[['close']].rename(columns={'close': 'Close'})


def fetch_yahoo_unadjusted(ticker, start_date, end_date):
    raw = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=False,
    )
    if raw is None or len(raw) == 0:
        raise ValueError(f"Yahoo returned no data for {ticker} between {start_date} and {end_date}")
    if hasattr(raw.columns, 'levels'):
        raw.columns = raw.columns.get_level_values(0)
    if 'Close' not in raw.columns:
        raise ValueError(f"Yahoo data for {ticker} has no Close column")
    df = raw[['Close']].copy()
    df.index = pd.DatetimeIndex(df.index).tz_localize(None)
    return df


def load_base_price(ticker):
    if ticker not in db_full.columns:
        raise ValueError(f"Ticker {ticker} not in base data")
    df = db_full[[ticker]].copy()
    df.index = pd.DatetimeIndex(df.index)
    return df.rename(columns={ticker: 'Close'})

# Validate unadjusted close prices using Yahoo data prior to 2016,
# comparing the base with both Yahoo and Tiingo for the overlapping dates.
VALIDATE_TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JNJ', 'XOM']

for ticker in VALIDATE_TICKERS:
    try:
        if ticker not in db_full.columns:
            print(f"{ticker}: not in base, skipping")
            continue

        base_df = load_base_price(ticker).dropna()
        if len(base_df) < 10:
            print(f"{ticker}: not enough base data, skipping")
            continue

        start_date = base_df.index.min().strftime("%Y-%m-%d")
        end_date = (base_df.index.max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

        yahoo_df = fetch_yahoo_unadjusted(ticker, start_date, end_date)
        tiingo_df = fetch_tiingo(ticker, start_date, end_date)

        common_dates = base_df.index.intersection(yahoo_df.index).intersection(tiingo_df.index)
        if len(common_dates) == 0:
            raise ValueError("No overlapping dates between base, Yahoo, and Tiingo")

        base_prices = base_df.loc[common_dates, 'Close']
        yahoo_prices = yahoo_df.loc[common_dates, 'Close']
        tiingo_prices = tiingo_df.loc[common_dates, 'Close']

        diff_yahoo_base = (yahoo_prices - base_prices).abs()
        diff_tiingo_base = (tiingo_prices - base_prices).abs()
        diff_yahoo_tiingo = (yahoo_prices - tiingo_prices).abs()

        print(f"{ticker}: {common_dates.min().date()} → {common_dates.max().date()} | rows {len(common_dates)}")
        print(f"  max(|Yahoo-Base|)   = {diff_yahoo_base.max():.4f}")
        print(f"  max(|Tiingo-Base|)  = {diff_tiingo_base.max():.4f}")
        print(f"  max(|Yahoo-Tiingo|) = {diff_yahoo_tiingo.max():.4f}")

        if diff_yahoo_base.max() > 0.01 or diff_tiingo_base.max() > 0.01:
            print(f"  WARNING: large difference on {ticker}")
            print(f"  Top Yahoo/Base mismatches:\n{diff_yahoo_base.sort_values(ascending=False).head(5)}")
    except Exception as e:
        print(f"{ticker}: ERROR — {e}")

AAPL: 1990-01-02 → 2015-12-24 | rows 6219
  max(|Yahoo-Base|)   = 112.0962
  max(|Tiingo-Base|)  = 610.0971
  max(|Yahoo-Tiingo|) = 677.0250
  Top Yahoo/Base mismatches:
2014-02-19    112.096214
2014-04-22    111.953214
2014-02-20    111.498957
2014-04-23    110.138328
2014-02-24    109.895130
Name: Close, dtype: float64
MSFT: 1990-01-02 → 2015-12-24 | rows 6219
  max(|Yahoo-Base|)   = 43.1469
  max(|Tiingo-Base|)  = 151.0660
  max(|Yahoo-Tiingo|) = 147.8906
  Top Yahoo/Base mismatches:
2000-01-03    43.14685
2000-01-05    40.00545
2000-01-04    39.58610
2000-01-07    38.64345
2000-01-18    38.29115
Name: Close, dtype: float64
GOOGL: 2004-08-19 → 2015-12-24 | rows 2646
  max(|Yahoo-Base|)   = 755.6680
  max(|Tiingo-Base|)  = 669.0800
  max(|Yahoo-Tiingo|) = 1189.6352
  Top Yahoo/Base mismatches:
2015-12-24    755.668000
2015-11-27    745.191501
2015-12-23    743.814501
2015-12-02    740.317498
2015-11-30    739.707498
Name: Close, dtype: float64
AMZN: 1997-05-15 → 2015-12-24 | rows 440